# Regularization Techniques in Deep Learning

**Learning Objectives:** Understand L1/L2 regularization, Dropout, Batch Normalization, Early Stopping, and data augmentation. Learn when and how to apply each technique.

## 1. The Overfitting Problem

Overfitting occurs when a model learns training data noise instead of the underlying pattern.

**Signs of overfitting:**
- Training loss → 0, validation loss increases
- Training accuracy near 100%, test accuracy significantly lower
- Model weights have extreme values

**Regularization** techniques mitigate overfitting by constraining the model.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np
import matplotlib.pyplot as plt

# Generate a small noisy dataset to demonstrate overfitting
np.random.seed(42)
X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = np.sin(X) + 0.3 * np.random.randn(100, 1)

# Use a small training set to encourage overfitting
X_train, y_train = X[:30], y[:30]
X_val, y_val = X[30:60], y[30:60]
X_test, y_test = X[60:], y[60:]

plt.scatter(X_train, y_train, label='Train', alpha=0.6)
plt.scatter(X_val, y_val, label='Val', alpha=0.6)
plt.legend()
plt.title('Noisy Sine Data')
plt.show()

## 2. L1/L2 Regularization

Adds a penalty to the loss function based on weight magnitudes.

| Type | Penalty | Effect |
|------|---------|--------|
| **L1 (Lasso)** | $\lambda \sum |w_i|$ | Sparse weights (feature selection) |
| **L2 (Ridge)** | $\lambda \sum w_i^2$ | Small weights (weight decay) |
| **ElasticNet** | $\lambda_1 \sum |w_i| + \lambda_2 \sum w_i^2$ | Combination |

In Keras: `kernel_regularizer=regularizers.l2(0.01)`

In [ ]:
# Build overfitting vs L2-regularized models
def build_model(reg=None):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(1,),
                     kernel_regularizer=reg),
        layers.Dense(128, activation='relu', kernel_regularizer=reg),
        layers.Dense(128, activation='relu', kernel_regularizer=reg),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

overfit_model = build_model()
l2_model = build_model(regularizers.l2(0.01))

hist_overfit = overfit_model.fit(X_train, y_train, epochs=500,
                                  validation_data=(X_val, y_val), verbose=0)
hist_l2 = l2_model.fit(X_train, y_train, epochs=500,
                        validation_data=(X_val, y_val), verbose=0)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_overfit.history['loss'], label='Train (no reg)')
plt.plot(hist_overfit.history['val_loss'], label='Val (no reg)')
plt.title('No Regularization')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist_l2.history['loss'], label='Train (L2)')
plt.plot(hist_l2.history['val_loss'], label='Val (L2)')
plt.title('L2 Regularization')
plt.legend()
plt.show()

## 3. Dropout

Dropout randomly deactivates a fraction of neurons during training, forcing the network to learn redundant representations.

- **Rate tuning:** Common rates: 0.2 (small models), 0.5 (large models)
- **At inference:** Dropout is disabled; weights are scaled by (1 - rate)
- **SpatialDropout:** Drops entire feature maps (for Conv layers)

In [ ]:
# Compare dropout rates
def build_dropout_model(rate=0.0):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(1,)),
        layers.Dropout(rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(rate),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

dropout_rates = [0.0, 0.2, 0.5]
histories = {}
for rate in dropout_rates:
    m = build_dropout_model(rate)
    h = m.fit(X_train, y_train, epochs=300, validation_data=(X_val, y_val), verbose=0)
    histories[rate] = h

plt.figure(figsize=(12, 4))
for rate in dropout_rates:
    plt.plot(histories[rate].history['val_loss'], label=f'Dropout {rate}')
plt.title('Validation Loss: Dropout Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. Batch Normalization

Batch Normalization normalizes layer inputs across the batch, reducing internal covariate shift.

$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma^2_B + \epsilon}}$

$y = \gamma \hat{x} + \beta$

**Benefits:**
- Allows higher learning rates
- Reduces dependence on initialization
- Acts as a regularizer (adds slight noise)
- Accelerates convergence

In [ ]:
# Batch Normalization on MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test = x_test.reshape(-1, 784).astype('float32') / 255.0
y_train_cat = keras.utils.to_categorical(y_train)
y_test_cat = keras.utils.to_categorical(y_test)

def build_mnist_model(use_batchnorm=False):
    model = keras.Sequential()
    model.add(layers.Dense(256, input_shape=(784,)))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dense(128))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_no_bn = build_mnist_model(False)
model_bn = build_mnist_model(True)

hist_no_bn = model_no_bn.fit(x_train, y_train_cat, epochs=10,
                              validation_split=0.2, verbose=0)
hist_bn = model_bn.fit(x_train, y_train_cat, epochs=10,
                        validation_split=0.2, verbose=0)

plt.plot(hist_no_bn.history['val_loss'], label='No BN')
plt.plot(hist_bn.history['val_loss'], label='With BN')
plt.title('BatchNorm Effect on Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Early Stopping & Model Checkpoint

**EarlyStopping:** Stop training when a monitored metric stops improving.

| Parameter | Description |
|-----------|-------------|
| `monitor` | Metric to watch (e.g., 'val_loss') |
| `patience` | Epochs to wait before stopping |
| `restore_best_weights` | Revert to best epoch |
| `min_delta` | Minimum improvement threshold |

**ModelCheckpoint:** Save model at best epoch.

In [ ]:
# Early stopping in action
model_es = build_mnist_model(True)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('best_mnist.keras',
                                     monitor='val_accuracy',
                                     save_best_only=True, verbose=1)
]

hist_es = model_es.fit(x_train, y_train_cat, epochs=50,
                        validation_split=0.2, callbacks=callbacks, verbose=0)

print(f"Stopped at epoch {len(hist_es.history['loss'])}")
test_acc = model_es.evaluate(x_test, y_test_cat, verbose=0)[1]
print(f"Test accuracy: {test_acc:.4f}")

## 6. Data Augmentation as Regularization

Data augmentation generates diverse training samples, reducing overfitting. Already covered in CNN notebook — key point: augmentation is especially important when data is limited.

## 7. Comparison of Regularization Techniques

| Technique | When to Apply | Strength | Hyperparameters |
|-----------|---------------|----------|-----------------|
| **L1** | Feature selection needed | Medium | `lambda` (0.001 - 0.1) |
| **L2** | General overfitting | Strong | `lambda` (0.0001 - 0.01) |
| **Dropout** | Large networks | Strong | `rate` (0.2 - 0.5) |
| **BatchNorm** | Deep networks, always | Moderate | — (learned) |
| **Early stopping** | Always | Moderate | `patience` (5-20) |
| **Data augmentation** | Image/audio data | Strong | Transform params |

In [ ]:
# Comprehensive comparison on MNIST (all techniques)
def build_regularized_model(use_l2=False, use_dropout=False, use_bn=False):
    model = keras.Sequential()
    model.add(layers.Dense(256, input_shape=(784,)))
    if use_bn:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    if use_dropout:
        model.add(layers.Dropout(0.3))

    model.add(layers.Dense(128,
                          kernel_regularizer=regularizers.l2(0.001) if use_l2 else None))
    model.add(layers.Activation('relu'))
    if use_dropout:
        model.add(layers.Dropout(0.3))

    model.add(layers.Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

models = {
    'No Reg': build_regularized_model(False, False, False),
    'Dropout': build_regularized_model(False, True, False),
    'L2': build_regularized_model(True, False, False),
    'BN+Dropout': build_regularized_model(False, True, True),
    'All': build_regularized_model(True, True, True)
}

results = {}
for name, model in models.items():
    h = model.fit(x_train, y_train_cat, epochs=10, validation_split=0.2, verbose=0)
    acc = model.evaluate(x_test, y_test_cat, verbose=0)[1]
    results[name] = (h, acc)
    print(f"{name:15s}: Test acc = {acc:.4f}")

plt.figure(figsize=(12, 4))
for name, (h, acc) in results.items():
    plt.plot(h.history['val_loss'], label=name)
plt.title('Validation Loss: Regularization Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Interview Problems

### Problem 1: Compare Overfitting With/Without Regularization

Train a deep network on a small subset of CIFAR-10 (100 samples) with and without regularization. Show that regularization significantly improves generalization.

### Problem 2: Find Optimal Dropout Rate

Train models with dropout rates [0.0, 0.1, 0.2, 0.3, 0.4, 0.5] on a dataset and determine the optimal rate. Plot test accuracy vs dropout rate.

### Problem 3: Early Stopping vs Full Training

Compare a model trained with early stopping vs one trained for a fixed large number of epochs. Show that early stopping prevents overfitting.

In [ ]:
# Solution: Problem 2 — Optimal dropout rate
rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
rate_accs = []

for rate in rates:
    m = keras.Sequential([
        layers.Dense(256, activation='relu', input_shape=(784,)),
        layers.Dropout(rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(rate),
        layers.Dense(10, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(x_train, y_train_cat, epochs=10, validation_split=0.2, verbose=0)
    acc = m.evaluate(x_test, y_test_cat, verbose=0)[1]
    rate_accs.append(acc)
    print(f"Dropout {rate:.1f}: Test acc = {acc:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(rates, rate_accs, 'bo-')
plt.xlabel('Dropout Rate')
plt.ylabel('Test Accuracy')
plt.title('Optimal Dropout Rate')
plt.grid(alpha=0.3)
plt.show()

best_rate = rates[np.argmax(rate_accs)]
print(f"Best dropout rate: {best_rate}")

## Summary

- **L1/L2 regularization** penalizes large weights (L1 for sparsity, L2 for weight decay)
- **Dropout** randomly drops neurons during training, preventing co-adaptation
- **Batch Normalization** normalizes layer inputs, allowing higher LR and faster convergence
- **Early Stopping** halts training when validation performance plateaus
- **Model Checkpoint** saves the best model during training
- **Data augmentation** generates diverse training samples
- **Combining techniques** often yields the best results
- Regularization is applied via `kernel_regularizer`, `Dropout` layer, `BatchNormalization` layer, and `callbacks`